# Fake Review Detection — Modelling and Demo

## Project objective

In this modelling notebook, we develop and compare machine learning models for detecting fake reviews.  
The project uses a cleaned and combined dataset containing reviews from multiple sources and review domains.

## What we do in this notebook

We follow a structured modelling approach:

1. Load the cleaned combined dataset.
2. Check whether the data is suitable for modelling.
3. Explore the distribution of labels, sources and topics.
4. Convert review text into numerical features with TF-IDF.
5. Train and compare multiple classification models.
6. Evaluate the models with accuracy, precision, recall and F1-score.
7. Analyse performance per topic and source.
8. Inspect wrong predictions.
9. Save the best model.
10. Create a small Streamlit demo.

This approach addresses the main feedback from the first version by using multiple datasets, comparing several models fairly and adding a working demo.


## 1. Imports

In this section, we import the required packages for modelling and evaluation.

We use:
- `pandas` and `numpy` for data handling;
- `matplotlib` for visualisation;
- `scikit-learn` for preprocessing, modelling and evaluation;
- `joblib` to save the final model;
- `json` to save information about the selected model.


In [ ]:
from pathlib import Path
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

DATA_DIR = Path("data/processed")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("Data folder:", DATA_DIR.resolve())
print("Model folder:", MODEL_DIR.resolve())

## 2. Loading the data

We load the cleaned dataset that was created during the data preparation phase.  
The dataset contains standardised review text, labels and metadata such as source and topic.

The modelling process uses:

- `clean_text` as the input text;
- `target` as the prediction target;
- `source` and `topic` for additional analysis.

If separate train and test files are available, we load them directly. Otherwise, we create a new train/test split from the combined dataset.


In [ ]:
train_path = DATA_DIR / "fake_reviews_train.csv"
test_path = DATA_DIR / "fake_reviews_test.csv"
combined_path = DATA_DIR / "combined_fake_review_dataset_cleaned.csv"

if train_path.exists() and test_path.exists():
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    print("Loaded existing train/test files.")

elif combined_path.exists():
    combined = pd.read_csv(combined_path)

    train_df, test_df = train_test_split(
        combined,
        test_size=0.2,
        random_state=RANDOM_STATE,
        stratify=combined["target"]
    )

    train_df.to_csv(train_path, index=False)
    test_df.to_csv(test_path, index=False)

    print("Created train/test files from the combined cleaned dataset.")

else:
    raise FileNotFoundError(
        "No processed dataset found. First run the data cleaning and combining notebook."
    )

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

### Why we use a train/test split

We split the data into training and test data to check whether the models can generalise to unseen reviews.  
The models learn patterns from the training data, while the test data is only used to evaluate the final predictions.


## 3. Data quality checks

Before training the models, we check whether the required columns are present and whether the target values are valid.

The expected target values are:

- `0 = real review`
- `1 = fake review`

These checks help prevent modelling errors and make sure all models use the same data structure.


In [ ]:
required_columns = {"clean_text", "target", "label", "source", "topic"}

missing_train = required_columns - set(train_df.columns)
missing_test = required_columns - set(test_df.columns)

if missing_train or missing_test:
    raise ValueError(
        f"Missing columns. Train missing: {missing_train}. Test missing: {missing_test}."
    )

train_df["clean_text"] = train_df["clean_text"].fillna("").astype(str)
test_df["clean_text"] = test_df["clean_text"].fillna("").astype(str)

train_df = train_df[train_df["clean_text"].str.len() > 0].copy()
test_df = test_df[test_df["clean_text"].str.len() > 0].copy()

train_df["target"] = train_df["target"].astype(int)
test_df["target"] = test_df["target"].astype(int)

print("Train labels:")
print(train_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

print("\nUnique target values in train:", sorted(train_df["target"].unique()))
print("Unique target values in test:", sorted(test_df["target"].unique()))

## 4. Exploratory overview

We briefly explore the modelling data before training the models.

This section shows:

- the distribution of real and fake reviews;
- the number of reviews per topic;
- the average review length per label.

This gives us a better understanding of the dataset and helps us interpret the model results later.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
train_df["label"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Label distribution in training data")
ax.set_xlabel("Label")
ax.set_ylabel("Number of reviews")
plt.xticks(rotation=0)
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
train_df["topic"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Training reviews per topic")
ax.set_xlabel("Topic")
ax.set_ylabel("Number of reviews")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

train_df["review_length"] = train_df["clean_text"].str.split().apply(len)
test_df["review_length"] = test_df["clean_text"].str.split().apply(len)

print("Review length statistics by label:")
display(train_df.groupby("label")["review_length"].describe().round(1))

## 5. Preparing input and target variables

We separate the input variable and the target variable.

The input variable is:

- `clean_text`: the cleaned review text.

The target variable is:

- `target`: whether the review is real or fake.

Because machine learning models cannot directly use raw text, the text is converted into numerical features later with TF-IDF.


In [ ]:
X_train = train_df["clean_text"]
X_test = test_df["clean_text"]

y_train = train_df["target"]
y_test = test_df["target"]

print("Training samples:", len(X_train))
print("Test samples:", len(X_test))

## 6. Evaluation setup

We define reusable evaluation functions so every model is assessed in the same way.

We evaluate the models using:

- **Accuracy**: the overall percentage of correct predictions.
- **Precision for fake reviews**: how many predicted fake reviews are actually fake.
- **Recall for fake reviews**: how many actual fake reviews are detected.
- **F1-score for fake reviews**: the balance between precision and recall.

For this project, F1-score is especially useful because fake review detection should consider both false alarms and missed fake reviews.


In [ ]:
def get_predictions(model, X):
    return model.predict(X)


def get_scores(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_fake": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_fake": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "f1_fake": f1_score(y_true, y_pred, pos_label=1, zero_division=0)
    }


def evaluate_model(name, model, X_test, y_test):
    y_pred = get_predictions(model, X_test)
    scores = get_scores(y_test, y_pred)
    scores["model"] = name

    print("=" * 80)
    print(name)
    print("=" * 80)
    print(classification_report(
        y_test,
        y_pred,
        target_names=["real", "fake"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=["real", "fake"])

    fig, ax = plt.subplots(figsize=(5, 4))
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(f"Confusion matrix — {name}")
    plt.show()

    return scores, y_pred


def run_grid_search(name, pipeline, param_grid, X_train, y_train):
    print("\n" + "=" * 80)
    print("Training:", name)
    print("=" * 80)

    grid = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring="f1",
        cv=3,
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    print("Best CV F1:", round(grid.best_score_, 4))
    print("Best parameters:", grid.best_params_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_

## 7. Model selection

We compare multiple classification models on the same dataset and train/test split.

The models included are:

- Dummy baseline
- Naive Bayes
- Logistic Regression
- Linear SVM
- kNN
- Decision Tree

The dummy baseline is included as a reference point.  
A model is only useful if it performs clearly better than this simple baseline.


### Why we compare several models

Different algorithms learn patterns in different ways.  
For example, Naive Bayes often works well for text classification, while SVM and Logistic Regression can perform strongly with TF-IDF features.

By comparing multiple models, we can make a better decision instead of relying on one algorithm only.


In [ ]:
model_configs = {
    "Dummy baseline": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=3000)),
        ("model", DummyClassifier(strategy="most_frequent"))
    ]),

    "Naive Bayes": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 1))),
        ("model", MultinomialNB(alpha=1.0))
    ]),

    "Logistic Regression": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 1))),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "Linear SVM": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=5000, ngram_range=(1, 1))),
        ("model", LinearSVC(
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ]),

    "kNN": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 1))),
        ("svd", TruncatedSVD(n_components=50, random_state=RANDOM_STATE)),
        ("norm", Normalizer()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "Decision Tree": Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", max_features=3000, ngram_range=(1, 1))),
        ("svd", TruncatedSVD(n_components=50, random_state=RANDOM_STATE)),
        ("model", DecisionTreeClassifier(
            max_depth=30,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=RANDOM_STATE
        ))
    ])
}

## 8. Training and evaluation

In this section, we train each model on the training data and evaluate it on the test data.

All models are compared using the same evaluation metrics.  
This makes the comparison fair and avoids judging models on different data or different criteria.

The results are stored in a summary table for easy comparison.


In [ ]:
trained_models = {}
results = []
test_predictions = {}

for name, model in model_configs.items():
    print("\n" + "=" * 80)
    print("Training:", name)
    print("=" * 80)

    model.fit(X_train, y_train)

    trained_models[name] = {
        "model": model,
        "best_params": "Fixed parameters, no GridSearchCV used",
        "best_cv_f1": np.nan
    }

    scores, y_pred = evaluate_model(name, model, X_test, y_test)

    scores["best_cv_f1"] = np.nan
    results.append(scores)
    test_predictions[name] = y_pred

results_df = pd.DataFrame(results)

results_df = results_df[
    ["model", "accuracy", "precision_fake", "recall_fake", "f1_fake", "best_cv_f1"]
].sort_values("f1_fake", ascending=False)

results_df.round(4)

## 9. Model comparison

We compare the model results and select the best-performing model.

The main metric we use is the **F1-score for fake reviews**, because this metric balances:

- how reliable fake predictions are;
- how many fake reviews are actually detected.

The best model is selected based on the test performance.


### Why F1-score is central

In fake review detection, accuracy can be misleading when one class is easier to predict than the other.  
F1-score is more informative because it combines precision and recall for fake reviews.

This makes it suitable for comparing how well the models detect the class that matters most in this project.


In [ ]:
display(results_df.round(4))

fig, ax = plt.subplots(figsize=(9, 5))
results_df.set_index("model")["f1_fake"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Model comparison based on F1-score for fake reviews")
ax.set_xlabel("F1-score fake")
ax.set_ylabel("Model")
plt.tight_layout()
plt.show()

best_model_name = results_df.iloc[0]["model"]
best_model = trained_models[best_model_name]["model"]

print("Best model based on test F1-score:", best_model_name)
print("Best parameters:")
print(trained_models[best_model_name]["best_params"])

## 10. Performance per topic and source

We do not only evaluate the overall score.  
We also check how the best model performs across different review domains.

This is important because the combined dataset contains reviews from multiple sources and topics.  
A good model should not only perform well on average, but should also be reasonably stable across different types of reviews.


In [ ]:
best_y_pred = test_predictions[best_model_name]

evaluation_df = test_df.copy()
evaluation_df["prediction"] = best_y_pred
evaluation_df["predicted_label"] = evaluation_df["prediction"].map({0: "real", 1: "fake"})
evaluation_df["correct"] = evaluation_df["target"] == evaluation_df["prediction"]

def grouped_metrics(df, group_col):
    rows = []

    for group_value, group_df in df.groupby(group_col):
        if group_df["target"].nunique() < 2:
            continue

        rows.append({
            group_col: group_value,
            "n_reviews": len(group_df),
            "accuracy": accuracy_score(group_df["target"], group_df["prediction"]),
            "precision_fake": precision_score(group_df["target"], group_df["prediction"], pos_label=1, zero_division=0),
            "recall_fake": recall_score(group_df["target"], group_df["prediction"], pos_label=1, zero_division=0),
            "f1_fake": f1_score(group_df["target"], group_df["prediction"], pos_label=1, zero_division=0)
        })

    return pd.DataFrame(rows).sort_values("f1_fake", ascending=False)

topic_metrics = grouped_metrics(evaluation_df, "topic")
source_metrics = grouped_metrics(evaluation_df, "source")

print("Performance per topic:")
display(topic_metrics.round(4))

print("Performance per source:")
display(source_metrics.round(4))

## 11. Error analysis

We inspect wrong predictions to better understand the limitations of the model.

We look at:

- **False positives**: real reviews predicted as fake.
- **False negatives**: fake reviews predicted as real.

This helps us explain where the model struggles and prevents the evaluation from being only descriptive.


### What the error analysis tells us

The error analysis shows examples where the model makes incorrect predictions.  
This is useful because it helps us move beyond only reporting scores and gives insight into possible weaknesses of the model.


In [ ]:
wrong_predictions = evaluation_df[evaluation_df["correct"] == False].copy()

false_positives = wrong_predictions[
    (wrong_predictions["target"] == 0) &
    (wrong_predictions["prediction"] == 1)
]

false_negatives = wrong_predictions[
    (wrong_predictions["target"] == 1) &
    (wrong_predictions["prediction"] == 0)
]

print("Total wrong predictions:", len(wrong_predictions))
print("False positives: real reviews predicted as fake:", len(false_positives))
print("False negatives: fake reviews predicted as real:", len(false_negatives))

print("\nExamples of false positives:")
display(false_positives[["clean_text", "label", "predicted_label", "source", "topic"]].head(5))

print("\nExamples of false negatives:")
display(false_negatives[["clean_text", "label", "predicted_label", "source", "topic"]].head(5))

## 12. Interpretation points

This section provides key points for interpreting the model results.

When discussing the results, we focus on:

- which model performs best;
- how much better it performs than the baseline;
- whether precision or recall is more important for this use case;
- whether the model performs consistently across topics and sources;
- what the error analysis shows about the model’s limitations.


In [ ]:
print(f"The best performing model is: {best_model_name}")
print()
print("Important points to discuss:")
print("- The baseline model shows what performance looks like without learning useful patterns.")
print("- F1-score is useful because fake-review detection depends on both precision and recall.")
print("- Precision for fake reviews shows how reliable fake predictions are.")
print("- Recall for fake reviews shows how many fake reviews are detected.")
print("- The per-topic and per-source results show whether performance is stable across domains.")
print("- Error analysis gives examples of reviews that are difficult for the model.")

## 13. Saving the best model

After comparing the models, we save the best-performing pipeline.

The saved pipeline includes both:

- the TF-IDF text vectorizer;
- the trained classification model.

This is useful because the exact same preprocessing and model can then be used in the Streamlit demo.


In [ ]:
model_path = MODEL_DIR / "best_fake_review_pipeline.pkl"
info_path = MODEL_DIR / "best_model_info.json"

joblib.dump(best_model, model_path)

model_info = {
    "best_model": best_model_name,
    "target_mapping": {
        "0": "real",
        "1": "fake"
    },
    "best_params": trained_models[best_model_name]["best_params"],
    "test_scores": results_df[results_df["model"] == best_model_name].iloc[0].to_dict()
}

with open(info_path, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=4)

print("Saved model to:", model_path.resolve())
print("Saved model info to:", info_path.resolve())

## 14. Streamlit demo

We create a small Streamlit application to demonstrate the model in practice.

The demo allows a user to:

1. Enter a review text.
2. Let the model process the text.
3. Receive a prediction: fake or real.

The app can be started from the terminal with:

```bash
streamlit run app.py
```

This demo addresses the deployment requirement and shows how the model can be used with new review texts.


In [ ]:
app_code = r'''import re
from pathlib import Path

import joblib
import numpy as np
import streamlit as st


MODEL_PATH = Path("models/best_fake_review_pipeline.pkl")


def clean_text(value):
    value = str(value).lower()
    value = re.sub(r"<.*?>", " ", value)
    value = re.sub(r"http\S+|www\S+", " ", value)
    value = re.sub(r"[^a-z0-9\s']", " ", value)
    value = re.sub(r"\s+", " ", value).strip()
    return value


st.set_page_config(
    page_title="Fake Review Detector",
    page_icon="🕵️",
    layout="centered"
)

st.title("Fake Review Detector")
st.write("Enter a review text and the model will predict whether it looks fake or real.")

if not MODEL_PATH.exists():
    st.error("Model file not found. First run the modelling notebook and save the best model.")
    st.stop()

model = joblib.load(MODEL_PATH)

review = st.text_area(
    "Review text",
    height=160,
    placeholder="Type or paste a review here..."
)

if st.button("Predict"):
    if not review.strip():
        st.warning("Please enter a review first.")
    else:
        cleaned_review = clean_text(review)
        prediction = model.predict([cleaned_review])[0]

        label = "fake" if prediction == 1 else "real"

        st.subheader("Prediction")
        st.write(f"The review is predicted as: **{label.upper()}**")

        if hasattr(model, "predict_proba"):
            probability = model.predict_proba([cleaned_review])[0]
            fake_probability = probability[1]
            st.write(f"Estimated fake probability: **{fake_probability:.2%}**")
        elif hasattr(model, "decision_function"):
            score = model.decision_function([cleaned_review])[0]
            st.write(f"Decision score: **{score:.3f}**")
            st.caption("A higher score means the review is more likely to be fake.")

        st.caption("This is a model prediction and should not be treated as absolute proof.")
'''

with open("app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

print("Created Streamlit app file: app.py")

## 15. Conclusion

This notebook presents a complete modelling workflow for fake review detection.

The main improvements are:

- we use a combined dataset instead of one single dataset;
- we compare multiple models on the same train/test split;
- we evaluate the models with more than only accuracy;
- we analyse performance per topic and source;
- we inspect wrong predictions;
- we save the best model;
- we create a Streamlit demo for practical use.

